## Ejercicio 1.1

### a) 
Una permutación uniforme de tamaño 𝑛 es una lista elegida al azar entre las 𝑛! permutaciones de 0,...,𝑛 −1, todas con la misma probabilidad. Con la función del código base que genera permutaciones uniformes, genere 6000 permutaciones de tamaño 𝑛 = 3 y arme una tabla con la frecuencia de cada una de las 6 posibles.

In [42]:
# Imports
import json
import numpy as np
from codigo_base import *

# Variables globales
rng = np.random.default_rng(4)

In [43]:
N = 6000   
posiblesPermutaciones = [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]
# Armamos la lista de permutaciones
frecuenciaPermutaciones = [0] * 6

# Contamos la frecuencia de cada permutación
for i in range (N):
    permutacion = permutacion_aleatoria(3, rng)
    frecuenciaPermutaciones[posiblesPermutaciones.index(permutacion)] += 1
    
print(frecuenciaPermutaciones)

[1003, 1021, 966, 983, 995, 1032]


In [44]:
# Calculamos varianza muestral
# ddof=1 asegura que sea la varianza muestral (divide por n - 1)
varianza_muestral = np.var(frecuenciaPermutaciones, ddof=1)

print(varianza_muestral)

588.8


i) ¿Qué distribución tiene la frecuencia de una permutación fija?

La distribucion de la frecuencia de una permutacion fija es una binomial. Lo podemos pensar como la probabilidad de que salga justo esa permutacion en los N=6000 ensayos con $p = 1/6$.

In [45]:
desvioEstandar1000 = np.sqrt(1000*(1/6)*(5/6))
desvioEstandar6000 = np.sqrt(6000*(1/6)*(5/6))
print("Desvio con N = 1000: ", desvioEstandar1000)
print("Desvio con N = 6000: ", desvioEstandar6000)

Desvio con N = 1000:  11.785113019775793
Desvio con N = 6000:  28.867513459481287


ii) ¿De qué tamaño esperaría que fueran sus diferencias con 1000?

La diferencia respecto a N=6000 es menor. 

In [46]:
diferencias = [0]*6
for i in range (6):
    diferencias[i] = abs(N/6 - frecuenciaPermutaciones[i])

print("Desvio real para N = 6000: ", diferencias)

Desvio real para N = 6000:  [3.0, 21.0, 34.0, 17.0, 5.0, 32.0]


iii) ¿Lo observado es compatible con que las 6 sean equiprobables?

Lo observado es compatible con que las 6 sean equiprobables. Al dejar una permutacion fija y preguntarse cuántas veces salió, tenemos un experimento de Bernouli (salió o no salió) con probabilidad 1/6. A lo largo de las N tiradas la suma de los aciertos sigue la distribucion de la binomial. 

Tambien observamos que e desvio real para N=6000 no supera en ningun caso a $2*\sigma$

### b)
Los algoritmos son deterministas. Explique por qué, aun así, 𝑆𝑛, 𝐶𝑛 y 𝑄𝑛 son variables aleatorias. Con los conteos del experimento uniforme, determine cuáles de los siguientes conteos varían de una lista a otra y cuáles son constantes: swaps y comparaciones de insertion sort, swaps y comparaciones de selection sort, comparaciones de quicksort. 



i) ¿Por qué son constantes?

$S_n, C_n$ y $Q_n$ son variables aleatorias debido a que su valor depende del input, que es una lista que se genera de una variable aleatoria. Citando a la consigna tenemos que "El azar entra por la entrada: si la lista a ordenar es elegida al azar, la cantidad de operaciones es una variable aleatoria"

Para ver cuales conteos varían, vamos a analizar la varianza muestral de estos conteos.

In [47]:
conteos = open("conteos.jsonl","r")

cantUniformes = 0
insertionSort = ([],[])
selectionSort = ([],[])
quickSort = []

# Buscamos todos los datos que necesitamos
for linea in conteos:
    datos = json.loads(linea)
    if datos["experimento"] == "uniforme":
        cantUniformes += 1
        if datos["algoritmo"] == "insertion":
            insertionSort[0].append(datos["swaps"])
            insertionSort[1].append(datos["comparaciones"])

        if datos["algoritmo"] == "selection":
            selectionSort[0].append(datos["swaps"])
            selectionSort[1].append(datos["comparaciones"])

        if datos["algoritmo"] == "quick_primero":
            quickSort.append(datos["comparaciones"])


varInsertionSwaps = np.var(insertionSort[0], ddof=1)
varInsertionComparaciones = np.var(insertionSort[1], ddof=1)
varSelectionSwaps = np.var(selectionSort[0], ddof=1)
varSelectionComparaciones = np.var(selectionSort[1], ddof=1)
varQuickPrimeroComparaciones = np.var(quickSort, ddof=1)

print("Varianza Insertion Sort Swaps: ", varInsertionSwaps)
print("Varianza Insertion Sort Comparaciones: ", varInsertionComparaciones)
print("Varianza Selection Sort Swaps: ", varSelectionSwaps)
print("Varianza Selection Sort Comparaciones: ", varSelectionComparaciones)
print("Varianza Quick Sort Primero Comparaciones: ", varQuickPrimeroComparaciones)

Varianza Insertion Sort Swaps:  27508.54618084042
Varianza Insertion Sort Comparaciones:  27418.472333916958
Varianza Selection Sort Swaps:  3.4130262631315653
Varianza Selection Sort Comparaciones:  0.0
Varianza Quick Sort Primero Comparaciones:  3532.4710652826416


ii) ¿Cuáles son constantes y cuales varían?

La varianza muestral de las comparaciones del selection sort es nula y la de los swaps es muy pequeña, esto significa que la primera es constante y la segunda es prácticamente constante. Tomamos la varianza muestral y confiamos en los resultados debido a que la cantidad de datos utilizados es grande.

iii) Justifique el valor de los conteos constantes. ¿Puede un conteo ser constante aunque la entrada sea aleatoria?

Si porque la cantidad de comparaciones depende del tamaño de la entrada y no de que tan desordenada este la lista. La cantidad de swaps depende de si tienen la misma cantidad de elementod desordenados que ordenados, por lo que si son similares, la varianza es baja.

### c) 

Con los conteos del experimento desorden, ¿qué valores toman $𝑆_{100}$, $𝐶_{100}$ y $𝑄_{100}$ sobre la lista ya ordenada (𝑘 = 0)? ¿Son variables aleatorias? Compare el valor de $𝑄_{100}$ con los que se obtienen sobre permutaciones uniformes.

In [48]:
lista_Ordenada = lista_desordenada(100, 0, rng)

S_100, C_100 = insertion_sort(lista_Ordenada)
_, Q_100 = quicksort(lista_Ordenada)

print("Swaps Insertion sort (S_100): ", S_100)
print("Comparaciones Insertion sort (C_100): ", C_100)
print("Comparaciones Quick sort (Q_100): ", Q_100)

Swaps Insertion sort (S_100):  0
Comparaciones Insertion sort (C_100):  99
Comparaciones Quick sort (Q_100):  4950


i) ¿Son variables aleatorias?

No, no son variables aleatorias. Recordemos que la aleatoreidad de las variables venía de lo aleatorio en el orden de las listas. Al fijar la lista, se pierde la aleatoreidad.

ii) Compare el valor de $𝑄_{100}$ con los que se obtienen sobre permutaciones uniformes.

In [49]:
lista_desordenada_uniforme = permutacion_aleatoria(100, rng)
_, Q_100 = quicksort(lista_desordenada_uniforme)
print("Comparaciones Quick sort (Q_100): ", Q_100)

Comparaciones Quick sort (Q_100):  613
